In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [4]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

len(documents)

72

In [7]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [8]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results=5
)

search_results

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [9]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [42]:
from homework_rag_helper import RAGBase

In [18]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [19]:
result = assistant.rag("How does the agentic loop keep calling the model until it stops?")

In [ ]:
print(result[0])
print(result[1])
print(result[1].input_tokens)



The loop keeps calling the model in a `while True` loop.

Each turn it:
1. sends the current `messages` to the model,
2. checks the response for any `function_call`,
3. runs the tool and appends the tool result to `messages`,
4. sets `has_function_calls = True` if there was a tool call.

At the end of the turn, if `has_function_calls == False`, it breaks out of the loop. So it stops when the model returns a response with no function calls, meaning it has given a final answer.
ResponseUsage(input_tokens=7110, input_tokens_details=InputTokensDetails(cached_tokens=6912), output_tokens=134, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7244)
7110


In [27]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

len(chunks)

295

In [28]:
index.fit(chunks)

In [43]:
assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [44]:
result = assistant.rag("How does the agentic loop keep calling the model until it stops?")

In [45]:
print(result[0])
print(result[1])
print(result[1].input_tokens)

It keeps a `while True` loop and tracks whether the model returned any `function_call` items.

- Each turn, it calls the model.
- If the output includes a `function_call`, the code runs the tool, adds the result back to `messages`, and sets `has_function_calls = True`.
- If there are no function calls in that turn, `has_function_calls` stays `False`, and the loop breaks.

So the stop condition is: **no function calls this turn means the agent is done**.
ResponseUsage(input_tokens=2293, input_tokens_details=InputTokensDetails(cached_tokens=1792), output_tokens=112, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2405)
2293


In [49]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [51]:
def search(query: str) -> dict[str, str]:

    return index.search(
        query,
        num_results=5,
    )

In [52]:
agent_tools = Tools()
agent_tools.add_tool(search)

agent_tools.get_tools()


[{'type': 'function',
  'name': 'search',
  'description': 'No description provided.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [53]:
instructions = """
You're a course teaching assistant.
Answer the student's question using the search tool.
Make multiple searches with different keywords before answering.
""".strip()

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [54]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received


In [55]:
result.cost

CostInfo(input_cost=Decimal('0.00590775'), output_cost=Decimal('0.0024975'), total_cost=Decimal('0.00840525'))

In [56]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nAnswer the student's question using the search tool.\nMake multiple searches with different keywords before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How does the agentic loop work, and how is it different from plain RAG?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"agentic loop vs RAG retrieval augmented generation agentic loop"}', call_id='call_N0jLTkk4bWer8N2H6eTwwbhi', name='search', type='function_call', id='fc_0d9973809001b871006a396d6e5ce0819bb9946a17cfa163b4', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"what is agentic loop in AI planning tool use retrieval"}', call_id='call_mkfcKiQyLMF3lIAE7uBpxNl5', name='search', type='function_call', id='fc_0d9973809001b871006a396d6e5cf0819bb4993d8026988cfc', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"plain RAG vs agen